# Cardiac Dataset Profiling: Complexity and Fairness Risk
This notebook complements the EDA by focusing on dataset difficulty metrics and fairness risk signals from profiling outputs.

**Primary question:** how hard is each dataset to model, and where are fairness risks likely to appear before training?

> Cross-reference: representation/distribution diagnostics are covered in the EDA notebook.

## 1. Setup and imports
Load minimal utilities for profile loading, complexity summaries, and warning-friendly analysis.

In [1]:
from __future__ import annotations
from pathlib import Path
import sys
import warnings
import pandas as pd

ROOT_DIR = Path.cwd().resolve()
if (ROOT_DIR / "configs").exists() is False:
    ROOT_DIR = ROOT_DIR.parents[0]
sys.path.append(str(ROOT_DIR / "src"))

from fairxai.notebook_utils import (
    load_domain_config,
    get_relevant_datasets,
    make_figure_path_builder,
    load_raw_datasets,
)
from fairxai.notebook_utils.profiling import (
    EXPECTED_COMPLEXITY_METRICS,
    find_profile_files,
    load_profiles,
    dataset_overview_rows,
    complexity_rows,
    complexity_missing_rows,
    group_complexity_rows,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

## 2. Configuration and paths
Resolve project paths and locate profiling outputs plus raw data for group-level complexity checks.

In [2]:
MEDICAL_AREA = "cardiac"
NOTEBOOK_TYPE = "profiling"
SAVE_FIGURES = True

cfg = load_domain_config(ROOT_DIR, MEDICAL_AREA)
schema_cfg = cfg["schema_cfg"]
RAW_DIR = cfg["raw_dir"]

RESULTS_DIR = ROOT_DIR / "results" / MEDICAL_AREA
PROFILE_DIR = RESULTS_DIR / "profiling"
PROFILE_GLOB = "*_data_profile.json"
DATASETS = get_relevant_datasets(schema_cfg, MEDICAL_AREA)
FIGURES_DIR, fig_path = make_figure_path_builder(ROOT_DIR, MEDICAL_AREA, NOTEBOOK_TYPE)

PROFILE_DIR, FIGURES_DIR

(PosixPath('/home/miguel/Desktop/Dissertacao/Code/FairXAI/results/cardiac/profiling'),
 PosixPath('/home/miguel/Desktop/Dissertacao/Code/FairXAI/notebooks/figures/cardiac'))

## 3. Load profiling artifacts
Load dataset profile JSON files and confirm available datasets before computing summaries.

In [3]:
profile_files = find_profile_files(PROFILE_DIR, PROFILE_GLOB, DATASETS)
profile_files

{'cleveland': PosixPath('/home/miguel/Desktop/Dissertacao/Code/FairXAI/results/cardiac/profiling/cleveland_data_profile.json'),
 'kaggle_heart': PosixPath('/home/miguel/Desktop/Dissertacao/Code/FairXAI/results/cardiac/profiling/kaggle_heart_data_profile.json'),
 'cardio70k': PosixPath('/home/miguel/Desktop/Dissertacao/Code/FairXAI/results/cardiac/profiling/cardio70k_data_profile.json')}

In [4]:
profiles = load_profiles(profile_files)
missing_profiles = [name for name in DATASETS if name not in profiles]
print("Loaded profiles:", sorted(profiles.keys()))
if missing_profiles:
    print("Missing profiles:", missing_profiles)

Loaded profiles: ['cardio70k', 'cleveland', 'kaggle_heart']


---
# Section 1: Profiling coverage and complexity diagnostics
---

## 4. Dataset overview (context only)
Keep this compact because descriptive distribution analysis is covered in the EDA notebook.

In [5]:
overview_df = dataset_overview_rows(profiles, DATASETS)
display(overview_df)

,dataset,samples,features,target_prevalence
0,cleveland,297,17,0.461279
1,kaggle_heart,918,15,0.553377
2,cardio70k,70000,15,0.499700


## 5. Complexity metric availability and warnings
Show available metrics now and warn about missing target metrics (warnings only; no hard failures).

In [6]:
complexity_df = complexity_rows(
    profiles=profiles,
    datasets=DATASETS,
    metrics=EXPECTED_COMPLEXITY_METRICS,
)
missing_df = complexity_missing_rows(
    profiles=profiles,
    datasets=DATASETS,
    expected_metrics=EXPECTED_COMPLEXITY_METRICS,
)

display(complexity_df)
display(missing_df)

if missing_df["missing_count"].gt(0).any():
    print("⚠️ Some planned complexity metrics are not available yet. Notebook continues with available metrics.")

,dataset,F2,F3,F4,N2,N3,N4,Raug,L1,L2,L3,T1,BayesImbalance,available_metrics
0,cleveland,0.209469,0.989899,None,None,0.420875,None,0.855219,None,0.141414,None,None,0.077441,6
1,kaggle_heart,0.252474,0.993464,None,None,0.343137,None,0.787582,None,0.221133,None,None,0.106754,6
2,cardio70k,0.400759,0.999857,None,None,0.388500,None,0.875500,None,0.292071,None,None,0.000600,6


,dataset,available_count,missing_count,available_metrics,missing_metrics,extra_metrics
0,cleveland,6,6,"BayesImbalance, F2, F3, L2, N3, Raug","F4, L1, L3, N2, N4, T1",-
1,kaggle_heart,6,6,"BayesImbalance, F2, F3, L2, N3, Raug","F4, L1, L3, N2, N4, T1",-
2,cardio70k,6,6,"BayesImbalance, F2, F3, L2, N3, Raug","F4, L1, L3, N2, N4, T1",-


⚠️ Some planned complexity metrics are not available yet. Notebook continues with available metrics.


## 6. Complexity by sensitive group (available metrics only)
Compute subgroup complexity with current implemented metrics and keep missing ones as warnings, not errors.

In [7]:
raw = load_raw_datasets(RAW_DIR, DATASETS)

group_frames = []
for dataset_name in DATASETS:
    frame = raw.get(dataset_name)
    if frame is None or frame.empty:
        continue
    group_df = group_complexity_rows(
        df=frame,
        dataset_name=dataset_name,
        target_col="heart_disease",
        sensitive_cols=["sex", "age_group"],
        metrics=["F2", "F3", "N3", "Raug", "L2", "BayesImbalance"],
        min_samples=50,
    )
    group_frames.append(group_df)

group_complexity_df = pd.concat(group_frames, ignore_index=True) if group_frames else pd.DataFrame()
display(group_complexity_df)

,dataset,attribute,group,n_samples,status,F2,F3,N3,Raug,L2,BayesImbalance,missing_metrics
0,cleveland,sex,0,96,ok,0.032207,0.927083,0.312500,0.708333,0.093750,0.479167,-
1,cleveland,sex,1,201,ok,0.255566,0.955224,0.388060,0.791045,0.164179,0.114428,-
2,cleveland,age_group,40-49,75,ok,0.286028,0.960000,0.333333,0.706667,0.053333,0.413333,-
3,cleveland,age_group,50-59,126,ok,0.382043,0.968254,0.412698,0.912698,0.134921,0.031746,-
4,cleveland,age_group,60-69,73,ok,0.142501,0.876712,0.534247,0.972603,0.178082,0.178082,-
5,cleveland,age_group,70+,6,skipped (n < 50),NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,cleveland,age_group,<40,17,skipped (n < 50),NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,kaggle_heart,sex,0,193,ok,0.174631,0.958549,0.248705,0.642487,0.139896,0.481865,-
8,kaggle_heart,sex,1,725,ok,0.215135,0.994483,0.343448,0.775172,0.222069,0.263448,-
9,kaggle_heart,age_group,40-49,223,ok,0.209803,0.982063,0.367713,0.825112,0.170404,0.165919,-


## 7. Cross-dataset difficulty ranking (current metrics)
Build a compact ranking from currently available metrics only; this is an operational guide for downstream baseline and fairness notebooks.

In [8]:
ranking_metrics = ["F2", "F3", "N3", "Raug", "L2"]
ranking_df = complexity_df[["dataset"] + ranking_metrics].copy()

for metric in ranking_metrics:
    ranking_df[f"rank_{metric}"] = ranking_df[metric].rank(ascending=False, method="average")

rank_cols = [f"rank_{metric}" for metric in ranking_metrics]
ranking_df["difficulty_score"] = ranking_df[rank_cols].mean(axis=1)
ranking_df = ranking_df.sort_values("difficulty_score", ascending=False).reset_index(drop=True)

display(ranking_df[["dataset", "difficulty_score"] + rank_cols])

,dataset,difficulty_score,rank_F2,rank_F3,rank_N3,rank_Raug,rank_L2
0,cleveland,2.4,3.0,3.0,1.0,2.0,3.0
1,kaggle_heart,2.4,2.0,2.0,3.0,3.0,2.0
2,cardio70k,1.2,1.0,1.0,2.0,1.0,1.0


## 8. Fairness risk hypotheses for downstream notebooks
Translate current complexity signals into testable expectations for baseline and fairness analysis stages.

In [9]:
risk_rows = []
for _, row in ranking_df.iterrows():
    risks = []
    if pd.notna(row["rank_N3"]) and row["rank_N3"] <= 1.5:
        risks.append("Higher equal-opportunity risk (high N3)")
    if pd.notna(row["rank_F2"]) and row["rank_F2"] <= 1.5:
        risks.append("Higher predictive-parity risk (high F2)")
    if pd.notna(row["rank_L2"]) and row["rank_L2"] <= 1.5:
        risks.append("Linear models likely insufficient (high L2)")
    risk_rows.append({
        "dataset": row["dataset"],
        "difficulty_score": row["difficulty_score"],
        "predicted_risks": "; ".join(risks) if risks else "No immediate high-risk flag",
    })

risk_summary_df = pd.DataFrame(risk_rows)
display(risk_summary_df)

,dataset,difficulty_score,predicted_risks
0,cleveland,2.4,Higher equal-opportunity risk (high N3)
1,kaggle_heart,2.4,No immediate high-risk flag
2,cardio70k,1.2,Higher predictive-parity risk (high F2); Linea...


## 9. Export-ready tables
Keep profiling outputs ready for dissertation text and later pipeline upgrades.

In [10]:
EXPORT_DIR = RESULTS_DIR / "profiling" / "notebook_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

overview_df.to_csv(EXPORT_DIR / "dataset_overview.csv", index=False)
complexity_df.to_csv(EXPORT_DIR / "complexity_metrics_available.csv", index=False)
missing_df.to_csv(EXPORT_DIR / "complexity_missing_warnings.csv", index=False)
group_complexity_df.to_csv(EXPORT_DIR / "group_complexity_available.csv", index=False)
ranking_df.to_csv(EXPORT_DIR / "dataset_difficulty_ranking.csv", index=False)
risk_summary_df.to_csv(EXPORT_DIR / "predicted_fairness_risks.csv", index=False)

print(f"Saved profiling tables to: {EXPORT_DIR}")

Saved profiling tables to: /home/miguel/Desktop/Dissertacao/Code/FairXAI/results/cardiac/profiling/notebook_exports
